In [0]:
%pip install databricks-ai-search
dbutils.library.restartPython()

In [0]:
import os, sys, importlib, mlflow, pandas as pd

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
folder = "/Workspace" + os.path.dirname(nb_path)
sys.path.append(folder)
importlib.invalidate_caches()

from exam_set import EXAM
from scorers import ALL_SCORERS

In [0]:
spark.createDataFrame(pd.DataFrame(EXAM)).write.mode("overwrite") \
    .saveAsTable("workspace.privacylens.eval_exam")

display(spark.table("workspace.privacylens.eval_exam").groupBy("category").count())

In [0]:
display(spark.sql("""
SELECT chunk_id, citation, LEFT(chunk_text, 400) AS preview
FROM workspace.privacylens.silver_chunks
WHERE chunk_id IN ('DPDP_10_1', 'DPDP_29_1', 'GDPR_8_1', 'GDPR_12_1')
ORDER BY chunk_id
"""))

In [0]:
MODEL_NAME = "workspace.privacylens.rag_model"
champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
champion_version = mlflow.MlflowClient().get_model_version_by_alias(MODEL_NAME, "champion").version
print("Evaluating model version", champion_version)

def predict_fn(question):
    out = champion.predict(pd.DataFrame({"question": [question]}))
    return {"answer": out.loc[0, "answer"], "sources": out.loc[0, "sources"]}

eval_data = [
    {
        "inputs": {"question": q["question"]},
        "expectations": {
            "expected_sources": q["expected_sources"],
            "expected_facts": q["expected_facts"],
            "should_refuse": q["should_refuse"],
            "category": q["category"],
            "id": q["id"],
        },
    }
    for q in EXAM
]

In [0]:
os.chdir("/tmp")  # same Git-folder fix as in 07_register

with mlflow.start_run(run_name=f"exam_v1_model_v{champion_version}"):
    results = mlflow.genai.evaluate(
        data=eval_data,
        predict_fn=predict_fn,
        scorers=ALL_SCORERS,
    )

for k, v in sorted(results.metrics.items()):
    print(f"{k:40} {v}")